# CMP611 Progress-2 Full Colab Run (GUE + Extra Dataset)

This notebook runs your full workflow:
1. Load `project.zip`
2. Install dependencies
3. Download GUE
4. Build low-data splits for 3 GUE tasks
5. Download extra dataset (`human_ocr_ensembl`) using `genomic-benchmarks`
6. Convert extra dataset to `sequence,label` CSV format
7. Train DNABERT-2 (smoke + main runs)
8. Aggregate and export results

> Recommended runtime: **GPU (T4)**

In [ ]:
# 1) Check runtime
import torch, platform, os
print('Python platform:', platform.platform())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU is not enabled. Go to Runtime > Change runtime type > GPU')

## 2) Upload `project.zip`
If `/content/project.zip` already exists, you can skip this cell.

In [ ]:
# 2) Upload project.zip (manual upload dialog)
from google.colab import files
import os

if not os.path.exists('/content/project.zip'):
    print('Please upload /content/project.zip now...')
    files.upload()

print('Exists:', os.path.exists('/content/project.zip'))

In [ ]:
# 3) Unzip project
import os, shutil, zipfile

if os.path.exists('/content/project'):
    shutil.rmtree('/content/project')

with zipfile.ZipFile('/content/project.zip', 'r') as zf:
    zf.extractall('/content')

print('Project folder exists:', os.path.exists('/content/project'))
print('Files:', os.listdir('/content/project')[:20])

In [ ]:
# 4) Install dependencies
%cd /content/project
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip uninstall -y peft || true
!python -m pip install -q genomic-benchmarks

In [ ]:
# 5) Environment check
%cd /content/project
!python scripts/check_environment.py

In [ ]:
# 6) Download GUE
%cd /content/project
!python scripts/download_gue.py --out-dir data/raw

In [ ]:
# 7) Build low-data splits for 3 GUE tasks
%cd /content/project

# You can edit ratios/seeds here
RATIOS = '0.01,0.10,1.0'
SEEDS  = '13'

# Promoter
!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/prom/prom_300_all \
  --output-root data/low_data/prog2/prom_300_all \
  --ratios $RATIOS \
  --seeds $SEEDS

# Core Promoter
!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/prom/prom_core_all \
  --output-root data/low_data/prog2/prom_core_all \
  --ratios $RATIOS \
  --seeds $SEEDS

# Splice
!python scripts/build_low_data_splits.py \
  --source-dir data/raw/GUE/splice/reconstructed \
  --output-root data/low_data/prog2/splice_reconstructed \
  --ratios $RATIOS \
  --seeds $SEEDS

In [ ]:
# 8) Download and prepare extra dataset: human_ocr_ensembl
%cd /content/project

from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from genomic_benchmarks.loc2seq import download_dataset

dest = Path('/content/gb_data')
dest.mkdir(parents=True, exist_ok=True)
download_dataset('human_ocr_ensembl', version=0, dest_path=dest, force_download=False)
root = dest / 'human_ocr_ensembl'

def load_split(split_name):
    rows = []
    for label_name, label_id in [('negative', 0), ('positive', 1)]:
        d = root / split_name / label_name
        for fp in d.glob('*.txt'):
            seq = fp.read_text().strip().upper()
            rows.append({'sequence': seq, 'label': label_id})
    return pd.DataFrame(rows)

train_df = load_split('train')
test_df  = load_split('test')

# Create dev from train (10%)
train_df, dev_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=13,
    stratify=train_df['label']
)

out = Path('/content/project/data/raw/extra/human_ocr_ensembl')
out.mkdir(parents=True, exist_ok=True)
train_df.to_csv(out / 'train.csv', index=False)
dev_df.to_csv(out / 'dev.csv', index=False)
test_df.to_csv(out / 'test.csv', index=False)

print('Saved:', out)
print('train/dev/test:', len(train_df), len(dev_df), len(test_df))
print(train_df.head())

In [ ]:
# 9) Build low-data splits for extra dataset
%cd /content/project
RATIOS = '0.01,0.10,1.0'
SEEDS  = '13'

!python scripts/build_low_data_splits.py \
  --source-dir data/raw/extra/human_ocr_ensembl \
  --output-root data/low_data/prog2/human_ocr_ensembl \
  --ratios $RATIOS \
  --seeds $SEEDS

## 10) Smoke test (quick sanity run)
Runs one short training to verify end-to-end pipeline before full grid.

In [ ]:
%cd /content/project
!python scripts/train_dnabert2.py \
  --model_name_or_path zhihan1996/DNABERT-2-117M \
  --data_path data/low_data/prog2/prom_core_all/r1_seed13 \
  --run_name smoke_prom_core_r1 \
  --output_dir outputs/prog2 \
  --seed 13 \
  --model_max_length 70 \
  --max_steps 120 \
  --per_device_train_batch_size 8 \
  --per_device_eval_batch_size 16 \
  --gradient_accumulation_steps 1 \
  --learning_rate 3e-5 \
  --warmup_steps 10 \
  --weight_decay 0.01 \
  --evaluation_strategy steps \
  --save_strategy steps \
  --eval_steps 60 \
  --save_steps 60 \
  --logging_steps 20 \
  --load_best_model_at_end True \
  --metric_for_best_model eval_f1_macro \
  --greater_is_better True \
  --save_total_limit 1 \
  --report_to none \
  --fp16 True \
  --use_lora False

## 11) Main runs (Progress-2 default grid)
Default grid in this notebook:
- tasks: promoter, core_promoter, splice, ocr_extra
- ratios: r1, r10, r100
- seed: 13

You can extend later to seeds 42/3407 and ratios 5/25.

In [ ]:
%cd /content/project
python - << 'PY'
import subprocess, os

MODEL = 'zhihan1996/DNABERT-2-117M'
BASE_OUT = 'outputs/prog2/main_runs'

tasks = [
    # task_name, low_data_root, model_max_length, max_steps
    ('promoter',      'data/low_data/prog2/prom_300_all',        300, 600),
    ('core_promoter', 'data/low_data/prog2/prom_core_all',        70, 600),
    ('splice',        'data/low_data/prog2/splice_reconstructed', 400, 600),
    ('ocr_extra',     'data/low_data/prog2/human_ocr_ensembl',    300, 400),
]

ratios = ['r1_seed13', 'r10_seed13', 'r100_seed13']

for task_name, root, max_len, max_steps in tasks:
    for r in ratios:
        data_path = f'{root}/{r}'
        run_name = f'{task_name}_{r}'
        out_dir  = f'{BASE_OUT}/{task_name}/{r}'
        os.makedirs(out_dir, exist_ok=True)

        cmd = [
            'python', 'scripts/train_dnabert2.py',
            '--model_name_or_path', MODEL,
            '--data_path', data_path,
            '--run_name', run_name,
            '--output_dir', out_dir,
            '--seed', '13',
            '--model_max_length', str(max_len),
            '--max_steps', str(max_steps),
            '--per_device_train_batch_size', '8',
            '--per_device_eval_batch_size', '16',
            '--gradient_accumulation_steps', '1',
            '--learning_rate', '3e-5',
            '--warmup_steps', '50',
            '--weight_decay', '0.01',
            '--evaluation_strategy', 'steps',
            '--save_strategy', 'steps',
            '--eval_steps', '200',
            '--save_steps', '200',
            '--logging_steps', '50',
            '--load_best_model_at_end', 'True',
            '--metric_for_best_model', 'eval_f1_macro',
            '--greater_is_better', 'True',
            '--save_total_limit', '1',
            '--report_to', 'none',
            '--fp16', 'True',
            '--use_lora', 'False'
        ]

        print('\nRUN:', ' '.join(cmd))
        subprocess.run(cmd, check=True)

print('\nAll main runs finished.')
PY

In [ ]:
# 12) Aggregate all eval_results.json into one CSV
%cd /content/project
python - << 'PY'
import json
from pathlib import Path
import pandas as pd

rows = []
root = Path('outputs/prog2')
for fp in root.rglob('eval_results.json'):
    d = json.loads(fp.read_text())
    parts = fp.parts
    run_name = parts[parts.index('results') + 1] if 'results' in parts else fp.parent.name
    rows.append({
        'run_name': run_name,
        'path': str(fp),
        'eval_f1_macro': d.get('eval_f1_macro'),
        'eval_auroc': d.get('eval_auroc'),
        'eval_auprc': d.get('eval_auprc'),
        'eval_matthews_correlation': d.get('eval_matthews_correlation'),
        'eval_accuracy': d.get('eval_accuracy'),
        'eval_loss': d.get('eval_loss'),
        'eval_runtime': d.get('eval_runtime'),
        'eval_samples_per_second': d.get('eval_samples_per_second'),
        'eval_steps_per_second': d.get('eval_steps_per_second'),
        'epoch': d.get('epoch'),
    })

df = pd.DataFrame(rows).sort_values('run_name')
out = Path('outputs/prog2_summary_all_runs.csv')
df.to_csv(out, index=False)
print('Saved:', out, '| rows:', len(df))
print(df.head(30))
PY

In [ ]:
# 13) Create quick pivot table for slides
%cd /content/project
python - << 'PY'
import pandas as pd

df = pd.read_csv('outputs/prog2_summary_all_runs.csv')

def parse_task_ratio(name):
    if name.startswith('core_promoter'):
        task = 'core_promoter'
    elif name.startswith('promoter'):
        task = 'promoter'
    elif name.startswith('splice'):
        task = 'splice'
    elif name.startswith('ocr_extra'):
        task = 'ocr_extra'
    else:
        task = 'other'

    ratio = 'na'
    for p in name.split('_'):
        if p.startswith('r') and p[1:].isdigit():
            ratio = p
            break
    return task, ratio

df[['task','ratio']] = df['run_name'].apply(lambda x: pd.Series(parse_task_ratio(x)))

pivot_f1 = df.pivot_table(index='task', columns='ratio', values='eval_f1_macro', aggfunc='mean')
pivot_auc = df.pivot_table(index='task', columns='ratio', values='eval_auroc', aggfunc='mean')

print('F1-macro pivot:')
print(pivot_f1)
print('\nAUROC pivot:')
print(pivot_auc)

pivot_f1.to_csv('outputs/prog2_f1_pivot.csv')
pivot_auc.to_csv('outputs/prog2_auroc_pivot.csv')
print('\nSaved: outputs/prog2_f1_pivot.csv, outputs/prog2_auroc_pivot.csv')
PY

In [ ]:
# 14) Zip outputs for easy download
%cd /content/project
!zip -r /content/progress2_outputs.zip outputs/prog2 outputs/prog2_summary_all_runs.csv outputs/prog2_f1_pivot.csv outputs/prog2_auroc_pivot.csv >/dev/null
print('Created: /content/progress2_outputs.zip')

## Optional Extension (after core results)
If you still have GPU time:
- Add ratios `0.05,0.25`
- Add seeds `42,3407`
- Add `frozen + linear` comparison path
